# 07 - Interactive Mapping in Jupyter: Capturing Clicks with iPyleaflet

## Overview

Earlier in the course we used **Folium** to create interactive maps. Folium is excellent for displaying spatial data and exporting polished maps. However, when we want **Python to respond to user interaction**, such as clicking on the map and saving coordinates, we need a different tool.

In this lesson we switch to **ipyleaflet**, which integrates directly with Jupyter notebooks and allows Python to react to map events like mouse clicks.

Using ipyleaflet, we will build a small interactive tool that:

- Displays a map
- Captures latitude and longitude when the user clicks
- Places a marker on the clicked location
- Saves the coordinates to a file
- Provides a button to clear saved points

This demonstrates how spatial visualization and user interaction can work together inside a notebook environment.

---

# Why Move from Folium to ipyleaflet?

## Folium

Folium is designed primarily for **map visualization**.

It is great for:

- adding markers and popups  
- displaying layers  
- creating choropleths  
- exporting standalone HTML maps  

But once the map is rendered, most interaction happens **inside JavaScript in the browser**, not in Python.

That means Python cannot easily respond to events like mouse clicks.

---

## ipyleaflet

ipyleaflet solves this problem.

It allows:

- Python to receive map click events
- dynamic updates to the map
- integration with Jupyter widgets
- building small interactive map tools inside notebooks

In short:

| Tool | Strength |
|-----|------|
| Folium | Display and presentation |
| ipyleaflet | Interaction and event handling |

---

# Additional Tool: ipywidgets

`ipyleaflet` is built on top of **ipywidgets**, which provides interactive controls in Jupyter.

Widgets allow us to create interface elements such as:

- buttons
- output displays
- layout controls

In this example we use widgets to:

- control the size of the map
- create a **Clear Saved Points** button
- display status messages

---

# The Complete Example

You can see the complete example in the next cell. If you want to scroll down 
  
---

# Step-by-Step Explanation

## 1. Import Required Libraries

```python
import json
from pathlib import Path
from ipyleaflet import Map, Marker, LayersControl, WidgetControl
import ipywidgets as widgets
```

These libraries allow us to:

- build the map
- place markers
- create widgets
- save data to files

---

## 2. Define Storage for Clicked Points

```python
OUTFILE = Path("clicked_points.json")
clicked_points = []
markers = []
```

We store:

- coordinates in a Python list
- map markers in a separate list
- the file path where coordinates are saved

---

## 3. Load Previously Saved Points

```python
if OUTFILE.exists():
    clicked_points = json.loads(OUTFILE.read_text())
```

If the file already exists, we reload earlier clicks so they persist across notebook runs.

---

## 4. Create the Map

```python
m = Map(
    center=(40.0, -99.0),
    zoom=5,
    layout=widgets.Layout(width='100%', height='700px')
)
```

Important parameters:

| Parameter | Meaning |
|---|---|
| center | initial map location |
| zoom | zoom level |
| layout | controls map size |

Unlike Folium, ipyleaflet uses **widget layout objects** to control width and height.

---

## 5. Restore Existing Markers

```python
for pt in clicked_points:
    marker = Marker(location=(pt["lat"], pt["lon"]))
    markers.append(marker)
    m.add(marker)
```

If points were saved earlier, we redraw their markers on the map.

---

## 6. Create Widgets

```python
output = widgets.Output()
clear_btn = widgets.Button(description="Clear Saved Points")
```

Widgets provide:

- an output display for messages
- a button for clearing points

---

## 7. Handle Map Click Events

```python
def handle_interaction(**kwargs):
    if kwargs.get("type") == "click":
```

ipyleaflet sends interaction data as keyword arguments.

When a click occurs we extract the coordinates:

```python
lat, lon = kwargs["coordinates"]
```

Then we:

1. store the coordinates
2. place a marker
3. save the data to a file
4. display a message

---

## 8. Clear Saved Points

The clear button triggers:

```python
clear_btn.on_click(clear_points)
```

This function:

- deletes stored points
- removes markers
- overwrites the saved file
- prints a message

---

## 9. Place the Button on the Map

```python
m.add(WidgetControl(widget=clear_btn, position="topright"))
```

This embeds the button inside the map interface.

---

## 10. Display the Map and Output Area

```python
display(m, output)
```

This shows:

- the interactive map
- the output log beneath it

---

## Key Concepts Learned

This example introduces several important ideas:

- **Event-driven programming**
  - The map responds to user interaction.

- **Persistent data storage**
  - User-generated data is saved to disk.

- **Widget-based interfaces**
  - Controls like buttons can interact with map logic.

- **Interactive spatial tools**
  - Maps can act as data collection interfaces.

---

## Summary

Using **ipyleaflet** and **ipywidgets**, we transformed a static map into an interactive spatial tool.

Users can now:

- click the map
- generate spatial data
- save the results
- manipulate the map with interface controls

These tasks that we implemented play a big part into our "Missile Geometry 101" final project. 

In [8]:
import json
from pathlib import Path
from ipyleaflet import GeoJSON, Map, Marker, LayersControl, WidgetControl
import ipywidgets as widgets


OUTFILE = Path("../data/clicked_points.json")
clicked_points = []
markers = []

if OUTFILE.exists():
    clicked_points = json.loads(OUTFILE.read_text())

m = Map(center=(40.0, -99.0), zoom=5,layout=widgets.Layout(width='100%', height='700px'))
m.add(LayersControl())


def save_points():
    OUTFILE.write_text(json.dumps(clicked_points, indent=2))

# Restore prior markers
for pt in clicked_points:
    marker = Marker(location=(pt["lat"], pt["lon"]))
    markers.append(marker)
    m.add(marker)

output = widgets.Output()
clear_btn = widgets.Button(description="Clear Saved Points")

def handle_interaction(**kwargs):
    if kwargs.get("type") == "click":
        lat, lon = kwargs["coordinates"]

        point = {"lat": round(lat, 6), "lon": round(lon, 6)}
        clicked_points.append(point)

        marker = Marker(location=(point["lat"], point["lon"]))
        markers.append(marker)
        m.add(marker)

        save_points()

        with output:
            print(f"Saved click: {point}")

def clear_points(_):
    clicked_points.clear()
    save_points()

    for marker in markers:
        try:
            m.remove(marker)
        except Exception:
            pass
    markers.clear()

    with output:
        print("Cleared saved points.")

clear_btn.on_click(clear_points)
m.on_interaction(handle_interaction)

m.add(WidgetControl(widget=clear_btn, position="topright"))

display(m, output)

Map(center=[40.0, -99.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

Output()

## Altering Marker Style

Changing the way things look is always something that either needs done, or we want to do. For this next quick lesson I want you to change the style of each marker that's placed on the map. 

- Copy the code from the last cell to a new python cell below.
- Use the example code below to change the icon of a marker. 
- There is a flag icon in the data folder, so the icon path should work.
- Look at the code to determine how to add your new marker to the map.
  
```python
from ipyleaflet import Marker, Icon

icon =  "../data/flag_icon.svg"

icon = Icon(icon_url=icon, icon_size=[30, 30])
marker = Marker(location=[40, -90], icon=icon)
```

In [ ]:
# Your Code Here